In [1]:
import os
os.environ["PYTHONHASHSEED"] = str(42)
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import torch
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses, util

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

/tmp/ipykernel_111479/1666899001.py:6: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses, util


## Load MS MARCO (English)

In [2]:
from datasets import load_dataset

ds_train = load_dataset("microsoft/ms_marco", "v1.1", split="train")
ds_val = load_dataset("microsoft/ms_marco", "v1.1", split="validation")

## Preprocess MS MARCO

### Extract query-positive-negative triplets

In [3]:
import random

def to_triplets(batch, num_neg=1):
    out = {"query": [], "positive": [], "negative": []}
    for q, p in zip(batch["query"], batch["passages"]):
        pos = [t for t, s in zip(p["passage_text"], p["is_selected"]) if s == 1]
        neg = [t for t, s in zip(p["passage_text"], p["is_selected"]) if s == 0]
        if not pos or not neg:
            continue                      # skip queries with no positive
        for n in random.sample(neg, min(num_neg, len(neg))):
            out["query"].append(q)
            out["positive"].append(pos[0])
            out["negative"].append(n)
    return out

random.seed(42)
train_triplets = ds_train.map(to_triplets, batched=True, remove_columns=ds_train.column_names)

random.seed(42)
val_triplets = ds_val.map(to_triplets, batched=True, remove_columns=ds_val.column_names)

In [4]:
train_triplets[0]

{'query': 'what is rba',
 'positive': 'Results-Based Accountability® (also known as RBA) is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole. RBA is also used by organizations to improve the performance of their programs. Creating Community Impact with RBA. Community impact focuses on conditions of well-being for children, families and the community as a whole that a group of leaders is working collectively to improve. For example: “Residents with good jobs,” “Children ready for school,” or “A safe and clean neighborhood”.',
 'negative': 'RBA uses a data-driven, decision-making process to help communities and organizations get beyond talking about problems to taking action to solve problems. It is a simple, common sense framework that everyone can understand. RBA starts with ends and works backward, towards means. The “end” or difference you are trying to make looks slightly d

### Adapt format to mMARCO script

The mMARCO scripts expects an input tsv of `<id> <sentence>`, which it then transforms into `<id> <translated sentence>`. So we extract the query, positive sequence and negative sequence from each triplet and save them into separate files with a generated id.

In [ ]:
import csv, re

def clean(t):
    return re.sub(r"[\t\r\n]+", " ", t).strip()

def export(ds, prefix):
    for col in ["query", "positive", "negative"]:
        with open(f"{prefix}_{col}.tsv", "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f, delimiter="\t")
            for i, text in enumerate(ds[col]):
                w.writerow([i, clean(text)])

export(train_triplets, "train")
export(val_triplets, "val")

### Translate

The resulting 6 files from the previous step are translated with the mMARCO script by:
1) activating the mmarco-env (see `mmarco-env-requirements.txt`)
2) running this from the path where the files are saved:
   
`for f in train_query train_positive train_negative val_query val_positive val_negative; do python ~/mMARCO_fork/scripts/translate.py --model_name_or_path Helsinki-NLP/opus-mt-en-sv --target_language sv --input_file $f.tsv --output_dir translated --batch_size 64 --max_seq_length 128 || break; done`

The resulting files are saved in the directory `translated/` with the prefix `translated_`.


### Re-construct triplets

In [6]:
from collections import defaultdict
from datasets import Dataset

def load(path):
    parts = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            i, s = line.rstrip("\n").split("\t", 1)
            parts[int(i)].append(s)
    return {i: " ".join(s) for i, s in parts.items()}

def rebuild(prefix, n):
    cols = {c: load(f"translated/translated_{prefix}_{c}.tsv") for c in ["query", "positive", "negative"]}
    keep = [i for i in range(n) if all(i in cols[c] for c in cols)]
    return Dataset.from_dict({c: [cols[c][i] for i in keep] for c in cols})

train_triplets_sv = rebuild("train", len(train_triplets))
val_triplets_sv = rebuild("val", len(val_triplets))

In [9]:
train_triplets_sv[0]

{'query': 'vad rba är',
 'positive': 'Resultatbaserad Accountability® (även känd som RBA) är ett disciplinerat sätt att tänka och vidta åtgärder som samhällen kan använda för att förbättra livet för barn, ungdomar, familjer, vuxna och samhället som helhet. RBA används också av organisationer för att förbättra prestandan hos deras program. Skapa gemenskapseffekter med RBA. Gemenskapens inverkan är inriktad på välfärdsvillkor för barn, familjer och hela samhället som en grupp ledare arbetar tillsammans för att förbättra. Till exempel: "Invånare med bra jobb", "Barn redo för skolan", eller "Ett säkert och rent område".',
 'negative': 'RBA använder en datadriven, beslutsfattande process för att hjälpa samhällen och organisationer att komma bortom att prata om problem för att vidta åtgärder för att lösa problem. Det är en enkel ram för sunt förnuft som alla kan förstå. RBA börjar med ändar och arbetar bakåt, mot medel. Den "slut" eller skillnad du försöker göra ser något annorlunda om du ar